In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
from joblib import Parallel, delayed
from tqdm import tqdm
API_KEY_2 = 'I7BU1yoA12r5RaYCvba8eC' # When we run out of keys
API_KEY = 'U2HIfMm7lzzOACIZH2GvPS' # Uffe
author_df = pd.read_csv('part1_authors_info.csv')

In [2]:
#There is no works count filter in the filtes, so we filter the works count before
#we find their works

mask = (author_df['works_count'] > 5) & (author_df['works_count'] < 5000)
author_df = author_df[mask]

In [4]:
WORKING_URL = 'https://api.openalex.org/works'
EMAIL = "s245109@dtu.dk"  

# 1. Definer emner og opsæt batches (præcis som du gjorde før)
social_ids = "C144024400|C15744967|C162324750|C17744445"
quant_ids = "C33923547|C121332964|C41008148"
BATCH_SIZE = 25

ids = author_df['author_id'].tolist() #
#Creating a list of all our Batches
batches = [ids[i : i + BATCH_SIZE] for i in range(0, len(ids), BATCH_SIZE)]

# 2. Lav en "Worker" funktion. Dette er koden der kører parallelt.
def fetch_works_for_batch(batch_ids):
    batch_idstr = '|'.join(batch_ids)
    current_cursor = '*'
    batch_works = [] # Lokal liste for denne specifikke batch
    
    filter_string = (
        f'author.id:{batch_idstr},'
        'authors_count:<10,'
        f'concepts.id:{social_ids},'
        f'concepts.id:{quant_ids},'
        'cited_by_count:>10'
    )

    while True:
        params = {
            'filter': filter_string,
            'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
            'per_page': 200,
            'cursor': current_cursor,
            'mailto': EMAIL,
            'api_key': API_KEY 
        }

        try:
            response = requests.get(WORKING_URL, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                meta = data.get('meta', {})
                
                if not results:
                    break
                    
                for work in results:
                    author_ids = [
                        auth['author']['id'].split('/')[-1] if auth.get('author') and auth['author'].get('id')
                        else auth['author']['id']
                        for auth in work.get('authorships', [])
                    ]
                    batch_works.append({
                        'id': work.get('id'),
                        'publication_year': work.get('publication_year'),
                        'cited_by_count': work.get('cited_by_count'),
                        'title': work.get('title'),
                        'author_ids': author_ids, 
                        'abstract_inverted_index': work.get('abstract_inverted_index')
                    })
                
                current_cursor = meta.get('next_cursor')
                if not current_cursor:
                    break
                    
            elif response.status_code == 429:
                # If we hit the API request limit.
                time.sleep(2)
                continue 
            else:
                #Errors log.
                print(f"ERROR: {response.status_code}")
                break
                
        except Exception as e:
            #Backup for the work.
            time.sleep(2)
            continue

    # Returner de værker, denne worker har fundet
    return batch_works

# 3. Kør Joblib Parallel processen
print(f"Starter parallel download for {len(batches)} batches...")

# n_jobs=4 betyder at den henter 4 batches PÅ SAMME TID. 
# tqdm() laver den flotte progress bar.
results_nested = Parallel(n_jobs=4)(
    delayed(fetch_works_for_batch)(batch) for batch in tqdm(batches, desc="Collecting works")
)

# 4. Flad listen ud
# Joblib returnerer en liste af lister (én liste per batch). Vi slår dem sammen til én lang liste.
all_works = [work for sublist in results_nested for work in sublist]

print(f"\nRetrieved {len(all_works)} works total.")

Starter parallel download for 33 batches...



Retrieved 10371 works total.


In [9]:
#Creating the dataframes
D1 = pd.DataFrame(all_works)

D1.to_csv('Ex3_get_all_works.csv')

print(D1.shape, f'we have found {D1.shape[0]} works') 

print('Empty cells:\n', D1.isna().sum())
print('We have decided to not remove/change any NaN labels')

D2 = D1[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D3 = D1[['id', 'title', 'abstract_inverted_index']]


D2.to_csv('Ex3_D2.csv')
D3.to_csv('Ex3_D3.csv')

(10371, 6) we have found 10371 works
Empty cells:
 id                            0
publication_year             12
cited_by_count                0
title                         1
author_ids                    0
abstract_inverted_index    2698
dtype: int64
We have decided to not remove/change any NaN labels


In [7]:
#Retrieving all the unique ID'

#Collecting all authors in a list
all_auth_ids = [author_id for sublist in D2['author_ids'] for author_id in sublist]

# Removing dupes via set.
unique_researchers = len(set(all_auth_ids))

print(unique_researchers)

16201


#### Dataset summaries:

**Dataset summary.** *How many works are listed in your Dataset D2 (IC2S2 papers) dataframe? How many unique researchers have co-authored these works?*




We have found **10371 works** with our filters,  as well a retrieveing **16201 unique authors**.


It's worth noting that this dataset hase some NaN values, see the section with NaN.
Many of these Nan's are attributed to the Abstract Inverted Index (AII) (2698 papers), but as the documentation notes is that quite common, as many older papers doesn't have an AII. 
Furthermore do we have one untitled paper and 12 papers with puplication years missing.

We have decided not to remove, or change any NaNs' yet, as we still could use the data to explore connectivity. (Author_Id's does NOT have any missing values, or empty lists)
Thus depending on the usecase these could be relevant. 


**Efficiency in code.** *Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?*


We optimized our code to maximize efficiency and minimize API requests using the following strategies:

1. Pre-filtering: Excluded authors outside the 5–5,000 works range before initiating any API calls.

2. Cursor Pagination: Utilized cursors to retrieve works efficiently per request.

3. Server-side Filtering: Applied complex API filters to download only relevant data.

4. Batching: Grouped 25 authors per request using the | (OR) operator, reducing total requests and conserving tokens.

5. Robustness: Implemented a pause condition to handle rate limits and prevent API overloads.

6. Parallel Processing: Executed requests concurrently across 4 cores. This step alone reduced the total execution time by a third.

We spent a lot of time and energy on making this code work well. We had a lot of issues with request overloading and getting the filters to work. It was a lot of work finding the methods in the documentation, while LLM's helped refining our solutions.

**Filtering Criteria and Dataset Relevance** *Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?*

The filters we used help find relevant CSS works and researchers. Filtering works per author removes large institutions and irrelevant authors. Requiring 10+ citations helps us look only at influential work, although this limit could be arbitrary. Excluding projects with >10 authors mitigates megaprojects that are beyond the scope of just CSS research and could dominate. Concept IDs are also crucial, as there is no accurate CSS label, so works should cover both subjects.

One flaw of our method is that it prioritizes older, established researchers while excluding new researchers and PhDs, as they lack citations and output. There could also be many CSS papers labeled under only one field, e.g., only Economics. Lastly, more niche subfields could be skipped, as smaller fields have fewer citations.